# SmolVLA Evaluation — Fixed

Fix: merge LoRA adapter into base model before eval (lerobot-eval CLI does not support raw LoRA adapters).


In [ ]:
# Cell 0: Install and setup
!pip install -q lerobot[smolvla,peft,libero] peft

import os
os.environ["MUJOCO_GL"] = "egl"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


In [ ]:
# Cell 1: Merge LoRA adapter into base model
# lerobot-eval cannot load raw LoRA adapters — must merge first
from peft import PeftModel
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
import torch

LORA_CHECKPOINT = "dennywu2966/smolvla-libero-object-lora"  # HF Hub
MERGED_PATH = "/kaggle/working/smolvla_merged"

print("Loading base SmolVLA...")
base_policy = SmolVLAPolicy.from_pretrained("lerobot/smolvla_base")

print(f"Loading LoRA adapter: {LORA_CHECKPOINT}")
lora_policy = PeftModel.from_pretrained(base_policy, LORA_CHECKPOINT)

print("Merging LoRA weights into base model...")
merged = lora_policy.merge_and_unload()

print(f"Saving merged model to {MERGED_PATH}")
merged.save_pretrained(MERGED_PATH)

peak_mem = torch.cuda.max_memory_allocated() / 1e9
print(f"Peak VRAM so far: {peak_mem:.2f} GB")
print("Merge complete.")


In [ ]:
# Cell 2: Run lerobot-eval on merged model
import subprocess, time

eval_env = {**os.environ, "MUJOCO_GL": "egl", "TOKENIZERS_PARALLELISM": "false"}

start = time.time()
proc = subprocess.Popen([
    "lerobot-eval",
    f"--policy.path={MERGED_PATH}",
    "--env.type=libero",
    "--env.task=libero_object",
    "--eval.batch_size=1",
    "--eval.n_episodes=20",
], env=eval_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

for line in proc.stdout:
    print(line, end="", flush=True)

proc.wait(timeout=10800)  # 3h timeout
elapsed = time.time() - start
print(f"\nEval exit code: {proc.returncode}")
print(f"Time: {elapsed/60:.1f} min")


In [ ]:
# Cell 3: Parse results and build comparison table
# lerobot-eval outputs JSON results to outputs/ directory
import json, glob
import sys
sys.path.insert(0, "/kaggle/working/vlm-vla/src")
from vlm_vla.eval_engine import EvalReport, TaskResult, compare_reports

# Find eval output JSON
result_files = glob.glob("outputs/*/eval_info.json") + glob.glob("outputs/eval_*.json")
print(f"Result files found: {result_files}")

if result_files:
    with open(sorted(result_files)[-1]) as f:
        raw = json.load(f)
    print("Raw eval output:")
    print(json.dumps(raw, indent=2)[:3000])
else:
    print("No result files found — check subprocess output above for errors")


In [ ]:
# Cell 4: Build comparison table
# Fill in values from raw eval output above
# Example structure — adapt task_names and success rates from Cell 3 output

# SmolVLA fine-tuned results (from Cell 3)
# smolvla_ft_results = [
#     TaskResult("libero_object_0", 0.80, 20, 150.0, {}),
#     ...  # add all 10 tasks
# ]

# SmolVLA zero-shot (known: all 0%)
smolvla_zs = EvalReport(
    model_name="SmolVLA-ZeroShot",
    task_suite="libero_object",
    results=[TaskResult(f"task_{i}", 0.0, 20, 400.0, {"timeout": 20}) for i in range(10)],
)

# OpenVLA 7B published reference
openvla_ref = EvalReport(
    model_name="OpenVLA-7B-FT",
    task_suite="libero_object",
    results=[TaskResult(f"task_{i}", 0.884, 20, 120.0, {}) for i in range(10)],
)

# TODO: replace with real smolvla_ft_results after running Cell 2-3
# smolvla_ft = EvalReport("SmolVLA-LoRA-FT", "libero_object", smolvla_ft_results)
# print(compare_reports(smolvla_ft, smolvla_zs, openvla_ref))

print("Zero-shot baseline:")
print(smolvla_zs.to_table())


In [ ]:
# Cell 5: Push merged model to HF Hub for persistence
# from huggingface_hub import HfApi
# api = HfApi()
# api.upload_folder(
#     folder_path=MERGED_PATH,
#     repo_id="dennywu2966/smolvla-libero-object-merged",
#     repo_type="model",
# )
# print("Merged model pushed to Hub!")
print("Uncomment above to push merged model to HF Hub")
